<a href="https://colab.research.google.com/github/byeonggyuk/nlp/blob/main/NLP_05%EB%B0%B0%ED%8F%AC%EC%9A%A9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5주차 실습: 텍스트 표현 방법

**주제:** Bag-of-Words(BoW), TF-IDF, n-gram 특징 추출

## 학습 목표
1. 토큰화된 텍스트를 수치 벡터로 표현해야 하는 이유를 설명한다.
2. CountVectorizer로 BoW 행렬을 만들고 희소행렬의 의미를 해석한다.
3. n-gram의 장점과 차원 증가·희소성의 한계를 판단한다.
4. TF-IDF의 TF·DF·IDF를 구분하고 과업에 맞게 설정한다.

> **제출:** 모든 TODO를 작성·실행하고 체크포인트 답변을 Markdown 셀에 남기세요. 외부 AI·자료를 사용했다면 사용 범위와 검증 과정을 기록하세요.

## 0. 준비
교육용 예시 데이터입니다. 실제 데이터는 출처·라이선스·개인정보(PII) 포함 여부를 확인하세요.

In [16]:
# 오류가 날 때만 아래 주석을 해제하세요.
!pip -q install -U scikit-learn pandas kiwipiepy

import pandas as pd
from kiwipiepy import Kiwi
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

kiwi = Kiwi()
corpus = [
    '배송이 진짜 번개 같고 발볼도 딱 맞아서 인생 스니커즈 등극함 ㅋㅋㅋ',
    '디자인이랑 색감은 미쳤는데 밑창이 딱딱해서 10분만 걸어도 발바닥 아파요',
    '사이즈 품절이라 한 치수 크게 샀더니 신발 안에서 발이 수영을 하네요',
    '가죽 재질도 부드럽고 가벼워서 데일리 운동화로 매일 신기 좋습니다',
    '유명 브랜드 제품이라 기대했는데 실밥 터져서 오고 마감 상태 눈물 나네요 ㅠㅠ'
]
df = pd.DataFrame({'doc_id': range(1, len(corpus)+1), 'text': corpus})
display(df)

,doc_id,text
0,1,배송이 진짜 번개 같고 발볼도 딱 맞아서 인생 스니커즈 등극함 ㅋㅋㅋ
1,2,디자인이랑 색감은 미쳤는데 밑창이 딱딱해서 10분만 걸어도 발바닥 아파요
2,3,사이즈 품절이라 한 치수 크게 샀더니 신발 안에서 발이 수영을 하네요
3,4,가죽 재질도 부드럽고 가벼워서 데일리 운동화로 매일 신기 좋습니다
4,5,유명 브랜드 제품이라 기대했는데 실밥 터져서 오고 마감 상태 눈물 나네요 ㅠㅠ


## 1. 텍스트를 벡터로 표현하는 이유
머신러닝 모델은 문장 자체가 아니라 숫자 입력을 계산합니다. 4주차의 형태소·토큰을 열(feature)로 만들고, 문서마다 각 토큰의 값(빈도 또는 중요도)을 채웁니다.

`원문 → 정제·토큰화 → 문서-토큰 행렬 → 분류/예측`

In [17]:
# TODO 1: 공백 기준 토큰화로 CountVectorizer를 만들고 문서-토큰 행렬을 확인하세요.
count_vec = CountVectorizer()
X_count = count_vec.fit_transform(corpus)
print(count_vec.get_feature_names_out())
display(pd.DataFrame(X_count.toarray(), columns=count_vec.get_feature_names_out()))

['10분만' 'ㅋㅋㅋ' 'ㅠㅠ' '가벼워서' '가죽' '같고' '걸어도' '기대했는데' '나네요' '눈물' '데일리' '등극함'
 '디자인이랑' '딱딱해서' '마감' '맞아서' '매일' '미쳤는데' '밑창이' '발바닥' '발볼도' '발이' '배송이' '번개'
 '부드럽고' '브랜드' '사이즈' '샀더니' '상태' '색감은' '수영을' '스니커즈' '신기' '신발' '실밥' '아파요'
 '안에서' '오고' '운동화로' '유명' '인생' '재질도' '제품이라' '좋습니다' '진짜' '치수' '크게' '터져서'
 '품절이라' '하네요']


,10분만,ㅋㅋㅋ,ㅠㅠ,가벼워서,가죽,같고,걸어도,기대했는데,나네요,눈물,...,인생,재질도,제품이라,좋습니다,진짜,치수,크게,터져서,품절이라,하네요
0,0,1,0,0,0,1,0,0,0,0,...,1,0,0,0,1,0,0,0,0,0
1,1,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,1,0,1,1
3,0,0,0,1,1,0,0,0,0,0,...,0,1,0,1,0,0,0,0,0,0
4,0,0,1,0,0,0,0,1,1,1,...,0,0,1,0,0,0,0,1,0,0


### 체크포인트 1
문서-토큰 행렬에서 **행**, **열**, **값**이 각각 무엇을 뜻하는지 1문장씩 설명하세요. 행렬의 0이 많은 이유도 설명하세요.

행 : 개별 문서
열 : 고유한 토큰
값 : 토큰이 등장한 빈도

0이 많은 이유 : 일부의 단어만 사용해서 빈도가 0으로 채워지기 때문

## 2. Bag-of-Words(BoW)
BoW는 각 단어의 등장 횟수를 사용합니다. 구현은 간단하지만 단어 순서와 긴 문맥 정보를 잃습니다. `배송이 빠르다`와 `빠르지 않은 배송`처럼 순서·부정이 중요한 경우를 생각해 보세요.

In [18]:
# TODO 2: max_features=5를 적용한 BoW를 만들고, 어떤 단어가 남았는지 확인하세요.
small_count_vec = CountVectorizer(max_features=5)
X_small = small_count_vec.fit_transform(corpus)
print(small_count_vec.get_feature_names_out())

['10분만' 'ㅋㅋㅋ' 'ㅠㅠ' '가벼워서' '가죽']


## 3. n-gram: 인접한 토큰 묶기
- unigram: `배송`, `빠르다`
- bigram: `배송 빠르다`
- trigram: `배송 이 빠르다`

n-gram은 일부 어순·표현 정보를 보완하지만, n이 커질수록 특징 수가 늘고 희소성이 커질 수 있습니다.

In [19]:
# TODO 3: unigram과 bigram을 함께 쓰는 CountVectorizer를 만드세요.
ngram_vec = CountVectorizer(ngram_range=(1, 2))
X_ngram = ngram_vec.fit_transform(corpus)
print(ngram_vec.get_feature_names_out())
print('특징 수:', X_ngram.shape[1])

['10분만' '10분만 걸어도' 'ㅋㅋㅋ' 'ㅠㅠ' '가벼워서' '가벼워서 데일리' '가죽' '가죽 재질도' '같고'
 '같고 발볼도' '걸어도' '걸어도 발바닥' '기대했는데' '기대했는데 실밥' '나네요' '나네요 ㅠㅠ' '눈물' '눈물 나네요'
 '데일리' '데일리 운동화로' '등극함' '등극함 ㅋㅋㅋ' '디자인이랑' '디자인이랑 색감은' '딱딱해서' '딱딱해서 10분만'
 '마감' '마감 상태' '맞아서' '맞아서 인생' '매일' '매일 신기' '미쳤는데' '미쳤는데 밑창이' '밑창이'
 '밑창이 딱딱해서' '발바닥' '발바닥 아파요' '발볼도' '발볼도 맞아서' '발이' '발이 수영을' '배송이' '배송이 진짜'
 '번개' '번개 같고' '부드럽고' '부드럽고 가벼워서' '브랜드' '브랜드 제품이라' '사이즈' '사이즈 품절이라' '샀더니'
 '샀더니 신발' '상태' '상태 눈물' '색감은' '색감은 미쳤는데' '수영을' '수영을 하네요' '스니커즈' '스니커즈 등극함'
 '신기' '신기 좋습니다' '신발' '신발 안에서' '실밥' '실밥 터져서' '아파요' '안에서' '안에서 발이' '오고'
 '오고 마감' '운동화로' '운동화로 매일' '유명' '유명 브랜드' '인생' '인생 스니커즈' '재질도' '재질도 부드럽고'
 '제품이라' '제품이라 기대했는데' '좋습니다' '진짜' '진짜 번개' '치수' '치수 크게' '크게' '크게 샀더니' '터져서'
 '터져서 오고' '품절이라' '품절이라 치수' '하네요']
특징 수: 95


### 체크포인트 2
감성 분석에서 bigram이 unigram보다 유용할 수 있는 표현을 하나 만들고, 그 이유를 설명하세요. 특징 수 증가가 항상 좋은 것은 아닌 이유도 함께 적으세요.

빠르지 않음 과 같은 표현의 경우 bigram에서는 부정의 표현을 판단할 수 있음. 이런 표현의 경우 unigram에서는 빠르지 와 않음 으로 분리되어 실질적 의미인 느리다 라는 것을 판단하지 못할 수 있음

## 4. TF-IDF: 많이 나오지만 덜 구별되는 단어의 가중치 낮추기
- **TF (Term Frequency):** 한 문서 안에서의 단어 빈도
- **DF (Document Frequency):** 해당 단어를 포함한 문서 수
- **IDF (Inverse Document Frequency):** 여러 문서에 흔한 단어의 가중치를 낮추는 값

TF-IDF는 단어의 중요도를 표현하는 한 방법이며, 과업에 따라 BoW가 더 적절할 수도 있습니다.

In [20]:
# TODO 4: TfidfVectorizer로 TF-IDF 행렬을 만들고 소수점 둘째 자리까지 확인하세요.
tfidf_vec = TfidfVectorizer()
X_tfidf = tfidf_vec.fit_transform(corpus)
tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf_vec.get_feature_names_out())
display(tfidf_df.round(2))

,10분만,ㅋㅋㅋ,ㅠㅠ,가벼워서,가죽,같고,걸어도,기대했는데,나네요,눈물,...,인생,재질도,제품이라,좋습니다,진짜,치수,크게,터져서,품절이라,하네요
0,0.00,0.32,0.00,0.00,0.00,0.32,0.00,0.00,0.00,0.00,...,0.32,0.00,0.00,0.00,0.32,0.00,0.00,0.00,0.00,0.00
1,0.33,0.00,0.00,0.00,0.00,0.00,0.33,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.32,0.32,0.00,0.32,0.32
3,0.00,0.00,0.00,0.33,0.33,0.00,0.00,0.00,0.00,0.00,...,0.00,0.33,0.00,0.33,0.00,0.00,0.00,0.00,0.00,0.00
4,0.00,0.00,0.29,0.00,0.00,0.00,0.00,0.29,0.29,0.29,...,0.00,0.00,0.29,0.00,0.00,0.00,0.00,0.29,0.00,0.00


In [21]:
# TODO 5: 아래 문서에서 BoW와 TF-IDF의 상위 3개 토큰을 비교하세요.
target_idx = 1 # '디자인이랑 색감은 미쳤는데 밑창이 딱딱해서 10분만 걸어도 발바닥 아파요'
count_scores = pd.Series(X_count.toarray()[target_idx], index=count_vec.get_feature_names_out())
tfidf_scores = pd.Series(X_tfidf.toarray()[target_idx], index=tfidf_vec.get_feature_names_out())
print('BoW 상위:', count_scores.sort_values(ascending=False).head(3))
print('TF-IDF 상위:', tfidf_scores.sort_values(ascending=False).head(3))

BoW 상위: 10분만     1
걸어도      1
디자인이랑    1
dtype: int64
TF-IDF 상위: 10분만     0.333333
걸어도      0.333333
디자인이랑    0.333333
dtype: float64


### 체크포인트 3
여러 문서에 공통으로 등장하는 단어는 TF-IDF에서 일반적으로 어떤 영향을 받나요? 이 단어가 과업상 중요할 수도 있는 사례를 하나 드세요.

df값이 커지고, idf값이 낮아집니다. 결국 tf-idf 중요도 점수는 낮게 나옵니다.

도메인 분류 및 특정 과업 식별에서 중요할 수 있습니다

## 5. 한국어 형태소 토큰을 벡터화하기
공백 토큰화 대신 4주차의 형태소 분석 결과를 벡터화할 수 있습니다. 여기서는 키워드 추출 예시로 명사 계열을 사용합니다. 감성 분석이라면 용언·부사·부정 표현 보존을 추가로 검토해야 합니다.

In [22]:
def kiwi_noun_tokens(text):
    return [token.form for token in kiwi.tokenize(text)
            if token.tag.startswith('N') and len(token.form) > 1]

print('예시 토큰:', kiwi_noun_tokens(corpus[0]))

# TODO 6: tokenizer에 kiwi_noun_tokens를 연결한 TfidfVectorizer를 만들고 결과를 확인하세요.
korean_tfidf = TfidfVectorizer(tokenizer=kiwi_noun_tokens, token_pattern=None)
X_korean = korean_tfidf.fit_transform(corpus)
print(korean_tfidf.get_feature_names_out())
display(pd.DataFrame(X_korean.toarray(), columns=korean_tfidf.get_feature_names_out()).round(2))

예시 토큰: ['배송', '번개', '인생', '스니커즈', '등극']
['가죽' '기대' '눈물' '데일리' '등극' '디자인' '마감' '밑창' '발바닥' '배송' '번개' '브랜드' '사이즈'
 '상태' '색감' '수영' '스니커즈' '신발' '실밥' '운동화' '유명' '인생' '재질' '제품' '치수' '품절']


,가죽,기대,눈물,데일리,등극,디자인,마감,밑창,발바닥,배송,...,스니커즈,신발,실밥,운동화,유명,인생,재질,제품,치수,품절
0,0.0,0.00,0.00,0.0,0.45,0.0,0.00,0.0,0.0,0.45,...,0.45,0.00,0.00,0.0,0.00,0.45,0.0,0.00,0.00,0.00
1,0.0,0.00,0.00,0.0,0.00,0.5,0.00,0.5,0.5,0.00,...,0.00,0.00,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00
2,0.0,0.00,0.00,0.0,0.00,0.0,0.00,0.0,0.0,0.00,...,0.00,0.45,0.00,0.0,0.00,0.00,0.0,0.00,0.45,0.45
3,0.5,0.00,0.00,0.5,0.00,0.0,0.00,0.0,0.0,0.00,...,0.00,0.00,0.00,0.5,0.00,0.00,0.5,0.00,0.00,0.00
4,0.0,0.35,0.35,0.0,0.00,0.0,0.35,0.0,0.0,0.00,...,0.00,0.00,0.35,0.0,0.35,0.00,0.0,0.35,0.00,0.00


## 6. 미니 과제: 특징 추출 정책 설계
다음 중 하나를 선택하세요. (1) 강의평 키워드 추출 (2) 상품 리뷰 감성 분석 (3) 학교 민원 자동 분류

분석기/토큰 단위, 품사 선택, n-gram 범위, 최소 문서 빈도(`min_df`), 사용할 표현(BoW 또는 TF-IDF)을 근거와 함께 결정하세요.

In [23]:
# TODO 7: 선택한 과업(상품 리뷰 감성 분석)에 맞게 아래 설정을 수정하고 실행하세요.

# 감성 분석을 위한 커스텀 토크나이저 정의
def kiwi_sentiment_tokens(text):
    tokens = kiwi.tokenize(text)
    result = []
    for token in tokens:
        # 명사, 동사, 형용사, 부사, 웹 문장 부호/이모티콘 및 기호 필터링
        if token.tag.startswith(('N', 'V', 'M', 'Z', 'U')) or token.form in ['안', '못', '않', 'ㅋㅋㅋ', 'ㅠㅠ']:
            result.append(token.form)
    return result

# 감성 분석용 벡터라이저 설정 (unigram + bigram, TF-IDF 사용)
my_vectorizer = TfidfVectorizer(
    tokenizer=kiwi_sentiment_tokens,
    token_pattern=None,
    ngram_range=(1, 2),
    min_df=1
)

# 벡터라이저 적용 및 특징 확인
X_my = my_vectorizer.fit_transform(corpus)
print('특징 수:', X_my.shape[1])
print('추출된 감성 특징 목록:\n', my_vectorizer.get_feature_names_out())
print('\nTF-IDF 행렬:')
display(pd.DataFrame(X_my.toarray(), columns=my_vectorizer.get_feature_names_out()).round(2))

특징 수: 103
추출된 감성 특징 목록:
 ['ㅋㅋㅋ' 'ㅠㅠ' '가볍' '가볍 데일리' '가죽' '가죽 재질' '같' '같 발' '걷' '걷 발바닥' '기대' '기대 실밥'
 '나' '나 ㅠㅠ' '눈물' '눈물 나' '데' '데 밑창' '데일리' '데일리 운동화' '등극' '등극 ㅋㅋㅋ' '디자인'
 '디자인 색감' '딱' '딱 맞' '마감' '마감 상태' '맞' '맞 인생' '매일' '매일 신' '미치' '미치 데' '밑창'
 '밑창 분' '발' '발 볼' '발 수영' '발바닥' '발바닥 아프' '배송' '배송 진짜' '번개' '번개 같' '볼' '볼 딱'
 '부드럽' '부드럽 가볍' '분' '분 걷' '브랜드' '브랜드 제품' '사' '사 신발' '사이즈' '사이즈 품절' '상태'
 '상태 눈물' '색감' '색감 미치' '수영' '수영 하' '스니커즈' '스니커즈 등극' '신' '신 좋' '신발' '신발 안'
 '실밥' '실밥 터지' '아프' '안' '안 발' '오' '오 마감' '운동화' '운동화 매일' '유명' '유명 브랜드' '이'
 '이 기대' '이 한' '인생' '인생 스니커즈' '재질' '재질 부드럽' '제품' '제품 이' '좋' '진짜' '진짜 번개'
 '치수' '치수 크' '크' '크 사' '터지' '터지 오' '품절' '품절 이' '하' '한' '한 치수']

TF-IDF 행렬:


,ㅋㅋㅋ,ㅠㅠ,가볍,가볍 데일리,가죽,가죽 재질,같,같 발,걷,걷 발바닥,...,치수 크,크,크 사,터지,터지 오,품절,품절 이,하,한,한 치수
0,0.21,0.0,0.00,0.00,0.00,0.00,0.21,0.21,0.00,0.00,...,0.00,0.00,0.00,0.0,0.0,0.00,0.00,0.00,0.00,0.00
1,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.24,0.24,...,0.00,0.00,0.00,0.0,0.0,0.00,0.00,0.00,0.00,0.00
2,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.21,0.21,0.21,0.0,0.0,0.21,0.21,0.21,0.21,0.21
3,0.00,0.0,0.24,0.24,0.24,0.24,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.0,0.0,0.00,0.00,0.00,0.00,0.00
4,0.00,0.2,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.2,0.2,0.00,0.00,0.00,0.00,0.00


### 체크포인트 4
선택한 과업의 특징 추출 정책을 설명하세요. 다음을 모두 포함합니다.
1. BoW/TF-IDF 선택 이유  2. 토큰·품사 선택  3. n-gram과 min_df 설정  4. 예상 오류 또는 정보 손실 사례

## 제출 체크리스트
- [ ] TODO 1~7을 모두 작성하고 실행했다.
- [ ] BoW 행렬의 행·열·값·희소성을 설명했다.
- [ ] unigram과 bigram의 차이·한계를 설명했다.
- [ ] TF·DF·IDF의 역할을 설명했다.
- [ ] 공백 기반과 형태소 기반 벡터화를 비교했다.
- [ ] 선택 과업에 맞는 특징 추출 정책과 오류 사례를 기록했다.
- [ ] 외부 AI/자료 사용 시 사용 범위와 검증 과정을 기록했다.

## 다음 주차 예고
다음 시간에는 만들어진 문서-토큰 행렬을 입력으로 사용하여 나이브 베이즈와 로지스틱 회귀 기반 텍스트 분류 모델을 학습하고 평가합니다.

### 체크포인트 4 답변

1. BoW/TF-IDF 선택 이유
: '진짜', '너무'와 같은 감성 강조어와 '안', '못' 등의 부정어, 그리고 개별 감성 형용사('좋다', '아쉽다' 등)의 상대적인 가중치를 포착하기 위해서.

2. 토큰 및 품사 선택
: 품사 범위 확장과 일반 키워드 추출과 달리, 감성 분석은 '좋다(형용사)', '환불하다(동사)', '안(부정어)', '매우(부사)' 등의 결합이 긍/부정 판단이 중요하기에

3. n-gram과 min_df 설정
ngram_range=(1, 2): '안' + '좋다'가 결합된 '안 좋다' 같은 부정 표현의 맥락을 누락 없이 파악하기 위해
min_df=1: 분석 대상 문서 수(5개)가 매우 적기 때문에, 단 한 번만 등장한 중요한 감성 단어가 누락되는 것을 방지하기 위해 최소 문서 빈도를 1로 설정

4. 예상 오류 또는 정보 손실 사례
: 어미 변화에 따른 단어 파편화: 현재 토크나이저는 형태소의 원형 복원처리가 완벽하지 않아 '느리'와 '느렸지만' 등이 서로 다른 토큰으로 취급되거나 결합되어 차원이 늘어날 수 있음
: 이중 부정의 한계:'나쁘지 않다'와 같이 unigram('나쁘')과 bigram('나쁘지 않다') 수준에서는 감성 파악이 되더라도, 더 길고 복잡한 문장 구조를 가진 부정 표현은 tri-gram 이상을 쓰지 않으면 부정어와 핵심 감성어의 결합 관계를 놓쳐 긍정으로 오분류될 수 있음

1. IDF 가 높은 단어일수록 전체 코퍼스 내에서 자주 등장하는 흔한 단어이다 ? (X)

2. BoW 의 가장 큰 한계점인 단어 순서 정보 유실 을 보완하기 위해 사용하는 기법은?
: n-gram 기법